In [0]:
%run ./save_utils

In [0]:
import json
import pyspark.sql.functions as F

def load_json_to_bronze(source_path, target_table):
    df = (spark.read
            .option("multiline", "true")
            .json(source_path)
            .select("*", "_metadata.file_path")
            .withColumnRenamed("file_path", "_source_file")
            .withColumn("_ingestion_timestamp", F.current_timestamp())
            # .withColumn("_source_file", F.input_file_name())
        )
    df.write.format("delta").mode("append").saveAsTable(target_table)


def load_data_to_bronze(endpoint_key):
    entity = endpoint_key.value
    target_dir = f"{PROFILE.get(ProfileKeys.LANDING_ROOT)}/{entity}"
    full_save_path = f"{target_dir}/*.json"
    target_table = PROFILE.get(ProfileKeys.BRONZE_ROOT).format(entity_name= entity)
    df = (spark.read
          .option("multiline", "true")
        #   .option("inferSchema", "true")
          .json(full_save_path)
        )
    display(df)
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(target_table)
    print(f"Data loaded to {target_table} from {full_save_path}")